Import thư viện và đọc dữ liệu

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Đọc dữ liệu California Housing
df = pd.read_csv("../data/california_housing.csv")

print("Kích thước dữ liệu:", df.shape)
display(df.head())
print("Số giá trị thiếu mỗi cột:")
print(df.isna().sum())
display(df.describe())

EDA

In [ ]:
# Khám phá phân phối biến mục tiêu và tương quan giữa các biến
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.histplot(data=df, x="MedHouseVal", bins=30, ax=axes[0])
axes[0].set_title("Phân phối MedHouseVal")
axes[0].set_xlabel("MedHouseVal (đơn vị 100.000 USD)")

sns.heatmap(df.corr(numeric_only=True), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=axes[1])
axes[1].set_title("Ma trận tương quan")

plt.tight_layout()
plt.savefig("figures/eda.png", dpi=300, bbox_inches="tight")
plt.show()

Tách X,Y .Chia dữ liệu 80% train 20% test

In [ ]:
# Tách đặc trưng và biến mục tiêu
features = ["MedInc", "HouseAge", "AveRooms", "AveBedrms", "Population", "AveOccup", "Latitude", "Longitude"]
X = df[features]
y = df["MedHouseVal"]

# Chia dữ liệu thành tập huấn luyện và kiểm tra
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Kích thước X_train:", X_train.shape)
print("Kích thước X_test:", X_test.shape)

Huấn luyện cây quyết định

In [ ]:
# Huấn luyện cây quyết định không giới hạn độ sâu làm baseline
baseline_model = DecisionTreeRegressor(random_state=42)
baseline_model.fit(X_train, y_train)

print(f"R2 trên tập train: {baseline_model.score(X_train, y_train):.4f}")
print(f"R2 trên tập test:  {baseline_model.score(X_test, y_test):.4f}")

Tìm tham số siêu tốc tốt nhất

In [ ]:
# Tìm siêu tham số tốt nhất bằng GridSearchCV
param_grid = {
    "max_depth": [5, 8, 10, 15],
    "min_samples_leaf": [10, 20, 30, 50]
}
grid_search = GridSearchCV(
    DecisionTreeRegressor(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring="neg_mean_squared_error"
)
grid_search.fit(X_train, y_train)

print("Tham số tốt nhất:", grid_search.best_params_)

So sánh baseline và mô hình tốt nhất trên tập train/test

In [ ]:
models = [
    ("Baseline", baseline_model),
    ("Tuned", grid_search.best_estimator_)
]
datasets = [("Train", X_train, y_train), ("Test", X_test, y_test)]
results = []

for model_name, model in models:
    for split_name, X_split, y_split in datasets:
        y_pred = model.predict(X_split)
        results.append({
            "Mô hình": model_name,
            "Tập dữ liệu": split_name,
            "MAE": mean_absolute_error(y_split, y_pred),
            "RMSE": np.sqrt(mean_squared_error(y_split, y_pred)),
            "R2": r2_score(y_split, y_pred)
        })

comparison_df = pd.DataFrame(results)
display(comparison_df)

In [ ]:
Trực quan hóa tầm quan trọng đặc trưng và dự đoán trên tập test

In [ ]:
best_model = grid_search.best_estimator_
importance = pd.Series(best_model.feature_importances_, index=features).sort_values()
y_test_pred = best_model.predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
importance.plot(kind="barh", ax=axes[0], color="steelblue")
axes[0].set_title("Feature importance")
axes[0].set_xlabel("Mức độ quan trọng")
axes[0].set_ylabel("Đặc trưng")

axes[1].scatter(y_test, y_test_pred, alpha=0.4)
value_min = min(y_test.min(), y_test_pred.min())
value_max = max(y_test.max(), y_test_pred.max())
axes[1].plot([value_min, value_max], [value_min, value_max], color="red")
axes[1].set_title("Giá thực tế và giá dự đoán")
axes[1].set_xlabel("Giá thực tế (100.000 USD)")
axes[1].set_ylabel("Giá dự đoán (100.000 USD)")
axes[1].set_xlim(value_min, value_max)
axes[1].set_ylim(value_min, value_max)

plt.tight_layout()
plt.savefig("figures/result.png", dpi=300, bbox_inches="tight")
plt.show()